In [1]:
%pylab inline

Populating the interactive namespace from numpy and matplotlib


In [2]:
import grid2op as gp

In [7]:
from datetime import timedelta


class PredictionChronicsHandler(gp.Chronics.ChronicsHandler):
    def __init__(self, 
                 current_datetime, 
                 current_step, 
                 maintenance, 
                 maintenance_time, 
                 maintenance_duration, 
                 load_p, 
                 load_q, 
                 prod_p, 
                 prod_v, 
                 time_interval=timedelta(minutes=5), 
                 max_iter=-1):
        # Properties
        self.max_iter = max_iter
        self.time_interval = time_interval
        
        # Current grid state
        self.n_line = 59
        self.current_datetime = current_datetime
        self.current_step = current_step
        # Maintenance
        self.maintenance = maintenance
        self.maintenance_time = maintenance_time
        self.maintenance_duration = maintenance_duration
#         self.maintenance_time = np.zeros(maintenance.shape, dtype='int') - 1
#         self.maintenance_duration = np.zeros(maintenance.shape, dtype='int')
#         for line_id in range(self.n_line):
#             self.maintenance_time[:, line_id] = self.get_maintenance_time_1d(self.maintenance[:, line_id])
#             self.maintenance_duration[:, line_id] = self.get_maintenance_duration_1d(self.maintenance[:, line_id])
        
        # Future chronics
        # TODO replace with predictions
        self.load_p = load_p
        self.load_q = load_q
        self.prod_p = prod_p
        self.prod_v = prod_v
        
        # XXX No hazards
        self.hazard_duration = np.zeros(self.n_line, dtype='int') - 1
        
    def next_time_step(self):
        self.current_datetime += self.time_interval
        self.current_step += 1
        
        return self.current_datetime, {
            'injection':{
                'load_p': self.load_p[self.current_step],
                'load_q': self.load_q[self.current_step],
                'prod_p': self.prod_p[self.current_step],
            },
            'maintenance': self.maintenance[self.current_step],
        }, \
        self.maintenance_time[self.current_step], \
        self.maintenance_duration[self.current_step], \
        self.hazard_duration, \
        self.prod_v[self.current_step]

    def done(self):
        if self.max_iter >= 0:
            return self.current_step >= self.max_iter
        else:
            return False


In [11]:
def sim_update_actions(env, handler):
    timestamp, tmp, maintenance_time, maintenance_duration, hazard_duration, prod_v = handler.next_time_step()
    if "injection" in tmp:
        env._injection = tmp["injection"]
    else:
        env._injection = None
    if 'maintenance' in tmp:
        env._maintenance = tmp['maintenance']
    else:
        env._maintenance = None
    if "hazards" in tmp:
        env._hazards = tmp["hazards"]
    else:
        env._hazards = None
    env.time_stamp = timestamp
    env._duration_next_maintenance = maintenance_duration
    env._time_next_maintenance = maintenance_time
    env._hazard_duration = hazard_duration
    return env.helper_action_env({"injection": env._injection, "maintenance": env._maintenance, "hazards": env._hazards}), prod_v

In [60]:
from lightsim2grid import LightSimBackend
backend = LightSimBackend()
env = gp.make('l2rpn_neurips_2020_track1_small', backend=backend)
env.reset()

In [5]:
null_ac = env.action_space({})

In [118]:
# Example: setup the simulator
pch = PredictionChronicsHandler(
    env.time_stamp,
    env.nb_time_step,
    env.chronics_handler.data.maintenance,
    env.chronics_handler.data.maintenance_time,
    env.chronics_handler.data.maintenance_duration,
    # Predictions
    env.chronics_handler.data.load_p,
    env.chronics_handler.data.load_q,
    env.chronics_handler.data.prod_p,
    env.chronics_handler.data.prod_v,
    max_iter=8062,
)
obs = env.get_obs()
obs.simulate(null_ac)
obs_env = obs._obs_env
obs_env.backend.initdc = False
obs_env._reset_to_orig_state()
obs_env._update_actions = lambda : sim_update_actions(obs_env, pch)
obs_env.chronics_handler = pch

In [125]:
obs_env.backend.initdc = False

In [123]:
sim_duration

8061

In [129]:
%%prun
obs_env._reset_to_orig_state()
obs_env.chronics_handler.current_step = env.nb_time_step
sim_done = False
sim_duration = 0
while not sim_done:
    sim_obs, sim_rew, sim_done, sim_info = obs_env.step(null_ac)
    sim_duration += 1
    if len(sim_info['exception']) > 0:
        print(sim_info)
# One rollout (with initdc set to False)
# 16 s ± 12.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

In [120]:
# One rollout should only take about 1s according to runpf estimate
8000*120/10**6

0.96

In [114]:
env.reward_helper.template_reward.alpha_redisph

5.0

In [110]:
env.gen_cost_per_MW

array([45.,  0., 46., 36., 48.,  0.,  0.,  0.,  0.,  0., 46.,  0.,  0.,
       44.,  0.,  0., 46.,  0.,  0., 35., 48., 40.], dtype=float32)

In [107]:
sim_obs.line_status

array([False,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True, False,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True])

In [80]:
obs_env.backend.get_line_status()

array([False,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True, False,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True])

In [124]:
sim_info

{'disc_lines': array([False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False,
        False, False, False, False, False]),
 'is_illegal': False,
 'is_ambiguous': False,
 'is_dispatching_illegal': False,
 'is_illegal_reco': False,
 'opponent_attack_line': None,
 'opponent_attack_sub': None,
 'opponent_attack_duration': 0,
 'exception': [],
 'rewards': {}}

In [93]:
env.opponent_action_class

grid2op.Action.PowerlineSetAction.PowerlineSetAction

In [68]:
sim_duration

8061

In [20]:
obs._obs_env._is_done(False, False)

False

In [63]:
env.reset()

In [61]:
done = False
duration = 0
while not done:
    obs, rew, done, info = env.step(null_ac)
    duration += 1
    if len(info['exception']) > 0:
        print(info)

{'disc_lines': array([False, False, False, False, False, False, False, False, False,
       False, False, False, False,  True, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
        True,  True, False,  True, False, False, False, False, False,
        True,  True,  True, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False]), 'is_illegal': False, 'is_ambiguous': False, 'is_dispatching_illegal': False, 'is_illegal_reco': False, 'opponent_attack_line': array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
        True, False, False, False, False, Fa

In [62]:
duration

622

In [54]:
env.time_stamp

datetime.datetime(2012, 4, 1, 6, 55)

In [44]:
obs_env.opponent_class

grid2op.Opponent.BaseOpponent.BaseOpponent

In [45]:
env.opponent_class

grid2op.Opponent.WeightedRandomOpponent.WeightedRandomOpponent

In [132]:
dqn_10k = [('/root/data_grid2op/l2rpn_neurips_2020_track1_small/chronics/Scenario_april_000',
   'Scenario_april_000',
   48371.640625,
   951,
   8062),
  ('/root/data_grid2op/l2rpn_neurips_2020_track1_small/chronics/Scenario_april_001',
   'Scenario_april_001',
   74869.875,
   1480,
   8062),
  ('/root/data_grid2op/l2rpn_neurips_2020_track1_small/chronics/Scenario_april_002',
   'Scenario_april_002',
   2369.6474609375,
   49,
   8062),
  ('/root/data_grid2op/l2rpn_neurips_2020_track1_small/chronics/Scenario_april_003',
   'Scenario_april_003',
   35416.19140625,
   687,
   8062),
  ('/root/data_grid2op/l2rpn_neurips_2020_track1_small/chronics/Scenario_april_004',
   'Scenario_april_004',
   39279.171875,
   759,
   8062),
  ('/root/data_grid2op/l2rpn_neurips_2020_track1_small/chronics/Scenario_april_005',
   'Scenario_april_005',
   9374.626953125,
   180,
   8062),
  ('/root/data_grid2op/l2rpn_neurips_2020_track1_small/chronics/Scenario_april_006',
   'Scenario_april_006',
   23330.837890625,
   446,
   8062),
  ('/root/data_grid2op/l2rpn_neurips_2020_track1_small/chronics/Scenario_april_007',
   'Scenario_april_007',
   42031.72265625,
   827,
   8062),
  ('/root/data_grid2op/l2rpn_neurips_2020_track1_small/chronics/Scenario_april_008',
   'Scenario_april_008',
   1590.54248046875,
   33,
   8062),
  ('/root/data_grid2op/l2rpn_neurips_2020_track1_small/chronics/Scenario_april_009',
   'Scenario_april_009',
   46388.95703125,
   895,
   8062)]

In [133]:
dqn_100k = [('/root/data_grid2op/l2rpn_neurips_2020_track1_small/chronics/Scenario_april_000',
   'Scenario_april_000',
   54756.953125,
   1079,
   8062),
  ('/root/data_grid2op/l2rpn_neurips_2020_track1_small/chronics/Scenario_april_001',
   'Scenario_april_001',
   3191.55078125,
   64,
   8062),
  ('/root/data_grid2op/l2rpn_neurips_2020_track1_small/chronics/Scenario_april_002',
   'Scenario_april_002',
   11733.6298828125,
   225,
   8062),
  ('/root/data_grid2op/l2rpn_neurips_2020_track1_small/chronics/Scenario_april_003',
   'Scenario_april_003',
   5169.1962890625,
   101,
   8062),
  ('/root/data_grid2op/l2rpn_neurips_2020_track1_small/chronics/Scenario_april_004',
   'Scenario_april_004',
   37368.47265625,
   717,
   8062),
  ('/root/data_grid2op/l2rpn_neurips_2020_track1_small/chronics/Scenario_april_005',
   'Scenario_april_005',
   35766.93359375,
   703,
   8062),
  ('/root/data_grid2op/l2rpn_neurips_2020_track1_small/chronics/Scenario_april_006',
   'Scenario_april_006',
   53979.5,
   1084,
   8062),
  ('/root/data_grid2op/l2rpn_neurips_2020_track1_small/chronics/Scenario_april_007',
   'Scenario_april_007',
   63058.0546875,
   1245,
   8062),
  ('/root/data_grid2op/l2rpn_neurips_2020_track1_small/chronics/Scenario_april_008',
   'Scenario_april_008',
   23775.904296875,
   455,
   8062),
  ('/root/data_grid2op/l2rpn_neurips_2020_track1_small/chronics/Scenario_april_009',
   'Scenario_april_009',
   46164.67578125,
   900,
   8062)]

In [140]:
mean([x[3] for x in dqn_10k])

630.7

In [141]:
mean([x[3] for x in dqn_100k])

657.3

In [142]:
median([x[2] for x in dqn_10k])

37347.681640625

In [144]:
median([x[2] for x in dqn_100k])

36567.703125